# KAN-24 - Hipoteses Estatisticas Antes dos Testes

**Projeto:** Spaceship Titanic - Analise Estatistica  
**Responsavel:** Daniela  
**Revisora dedicada:** Renatta  
**Base analitica:** `data/processed/spaceship_titanic_clean.csv`

## Objetivo desta etapa

Antes de executar qualquer teste inferencial, registramos as perguntas, as hipoteses e a regra de decisao. Esta ordem e importante: se uma hipotese fosse escolhida depois de observar os resultados, poderiamos favorecer explicacoes que parecem significativas apenas por acaso. Essa pratica e conhecida como *p-hacking*.

Esta entrega **nao calcula testes, valores-p, intervalos de confianca, tamanhos de efeito ou conclusoes**. Esses resultados pertencem as tarefas KAN-25, KAN-26 e KAN-27.

## 1. Preparacao reproduzivel

Usaremos apenas a base de treinamento tratada no KAN-12. Ela contem o desfecho observado `Transported` e as variaveis derivadas, incluindo `TotalSpend`. A base `test.csv` do Kaggle nao possui o desfecho e, por isso, nao serve para estes testes de hipoteses.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Permite executar o notebook a partir da raiz do projeto ou da pasta notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'spaceship_titanic_clean.csv'
alpha = 0.05

df = pd.read_csv(DATA_PATH)

print(f'Base carregada: {DATA_PATH}')
print(f'Dimensao: {df.shape[0]:,} passageiros x {df.shape[1]} colunas')
print(f'Nivel de significancia pre-especificado: alpha = {alpha:.2f}')

Base carregada: C:\dev\PosGraduacao\Estatistica\Trabalho Final\spaceship-titanic-estatistica-posdl\data\processed\spaceship_titanic_clean.csv
Dimensao: 8,693 passageiros x 22 colunas
Nivel de significancia pre-especificado: alpha = 0.05


## 2. Contrato dos dados para a inferencia

Antes de formular testes, confirmamos que as variaveis previstas existem, nao possuem ausencias e apresentam as categorias esperadas. Esta verificacao nao e um teste de hipotese: e uma protecao contra analisar a base errada ou uma versao incompleta dela.

In [2]:
REQUIRED_COLUMNS = {'HomePlanet', 'CryoSleep', 'TotalSpend', 'Transported'}
EXPECTED_PLANETS = {'Earth', 'Europa', 'Mars'}

missing_columns = REQUIRED_COLUMNS - set(df.columns)
if missing_columns:
    raise ValueError(f'Colunas obrigatorias ausentes: {sorted(missing_columns)}')

null_counts = df[list(REQUIRED_COLUMNS)].isna().sum()
if (null_counts > 0).any():
    raise ValueError(f'Valores ausentes nas variaveis analiticas: {null_counts[null_counts > 0].to_dict()}')

transported_categories = set(df['Transported'].unique())
if len(transported_categories) != 2:
    raise ValueError(f'Transported deve ter duas categorias; encontrado: {transported_categories}')

observed_planets = set(df['HomePlanet'].unique())
if observed_planets != EXPECTED_PLANETS:
    raise ValueError(
        f'Categorias inesperadas em HomePlanet. Esperado: {EXPECTED_PLANETS}; observado: {observed_planets}'
    )

print('Contrato validado: as quatro variaveis estao completas e disponiveis.')
print('\nContagem por HomePlanet:')
display(df['HomePlanet'].value_counts().rename_axis('HomePlanet').to_frame('n'))
print('\nContagem por CryoSleep:')
display(df['CryoSleep'].value_counts().rename_axis('CryoSleep').to_frame('n'))
print('\nContagem por Transported:')
display(df['Transported'].value_counts().rename_axis('Transported').to_frame('n'))

Contrato validado: as quatro variaveis estao completas e disponiveis.

Contagem por HomePlanet:


,n
HomePlanet,
Earth,4715
Europa,2174
Mars,1804



Contagem por CryoSleep:


,n
CryoSleep,
False,5558
True,3135



Contagem por Transported:


,n
Transported,
True,4378
False,4315


## 3. Conceitos essenciais

Uma **hipotese nula** ($H_0$) representa a ausencia de associacao ou diferenca. A **hipotese alternativa** ($H_1$) representa uma associacao ou diferenca que desejamos investigar.

O nivel de significancia foi fixado em $\alpha = 0{,}05$. Isso limita a 5% o risco planejado de rejeitar $H_0$ quando ela e verdadeira, o chamado **erro do tipo I**. Depois que os testes forem executados, a regra sera:

- se $p < 0{,}05$, rejeitar $H_0$;
- se $p \geq 0{,}05$, nao rejeitar $H_0$.

O valor-p nao e a probabilidade de $H_0$ ser verdadeira. Ele mede quao compativeis os dados observados seriam com um modelo em que $H_0$ fosse verdadeira. Por isso, **nao rejeitar** $H_0$ nao prova igualdade entre os grupos: significa apenas que os dados nao forneceram evidencia suficiente contra ela sob este criterio.

## 4. Pergunta 1 - Planeta de origem e teletransporte

**Pergunta de negocio:** a proporcao de passageiros teletransportados varia conforme o planeta de origem?

- **Variaveis:** `HomePlanet` (Earth, Europa, Mars) e `Transported` (True, False).
- **Unidade de analise:** um passageiro.
- **$H_0$:** `HomePlanet` e `Transported` sao independentes; a distribuicao de `Transported` e a mesma nos planetas de origem.
- **$H_1$:** existe associacao entre `HomePlanet` e `Transported`; pelo menos uma distribuicao/proporcao difere.
- **Teste previsto:** qui-quadrado de independencia, na KAN-25.

O qui-quadrado e adequado porque ambas as variaveis sao categoricas. Ele compara as frequencias observadas em uma tabela de contingencia com as frequencias que seriam esperadas se nao houvesse associacao.

## 5. Pergunta 2 - Criossomo e teletransporte

**Pergunta de negocio:** a proporcao de passageiros teletransportados varia entre quem estava ou nao em criossomo?

- **Variaveis:** `CryoSleep` (True, False) e `Transported` (True, False).
- **Unidade de analise:** um passageiro.
- **$H_0$:** `CryoSleep` e `Transported` sao independentes; a distribuicao de `Transported` e a mesma nos dois estados de criossomo.
- **$H_1$:** existe associacao entre `CryoSleep` e `Transported`; as distribuicoes/proporcoes diferem.
- **Teste previsto:** qui-quadrado de independencia, na KAN-25.

Tambem temos duas variaveis categoricas. O teste respondera se a diferenca observada nas frequencias e grande demais para ser atribuida apenas a variacao amostral sob a hipotese de independencia.

## 6. Pergunta 3 - Gasto total e teletransporte

**Pergunta de negocio:** passageiros teletransportados e nao teletransportados apresentam diferenca no gasto total a bordo?

- **Variaveis:** `TotalSpend` (quantitativa continua/discreta truncada em zero) e `Transported` (categorica binaria: True vs False).
- **Unidade de analise:** um passageiro.
- **Objetivo analitico:** comparar o comportamento de consumo entre quem foi e nao foi teletransportado. Sob a otica economica e financeira do negocio, o foco primario esta na diferenca de volume medio gasto ($\mu$), enquanto a otica distributiva analisa a tendencia central e ordenacao dos postos frente a forte assimetria e presenca massiva de zeros.

**Hierarquia de testes e hipoteses pre-registradas:**

1. **Teste Principal (orientador da conclusao de negocio): Teste t de Welch**
   - **Foco:** diferenca entre as medias populacionais ($\mu_1 - \mu_2$).
   - **$H_0$ (nula):** $\mu_{\text{Transported=True}} = \mu_{\text{Transported=False}}$ (o gasto medio a bordo e igual entre passageiros teletransportados e nao teletransportados).
   - **$H_1$ (alternativa):** $\mu_{\text{Transported=True}} \neq \mu_{\text{Transported=False}}$ (o gasto medio a bordo difere entre os dois grupos).
   - **Justificativa metodologica:** com $N = 8.693$ observacoes (mais de 4.300 por grupo), o Teorema Central do Limite (TCL) assegura a normalidade assintotica da distribuicao amostral das medias. A versao de Welch (`scipy.stats.ttest_ind(..., equal_var=False)`) e rigorosamente apropriada por nao exigir homocedasticidade (igualdade de variancias), acomodando eventuais dispersoes desiguais apontadas pelo Teste de Levene.

2. **Teste Complementar (robustez nao parametrica de postos): Teste Mann-Whitney U**
   - **Foco:** dominancia estocastica e diferenca na distribuicao de postos / medianas.
   - **$H_0$ (nula):** $P(X_{\text{True}} > Y_{\text{False}}) = P(Y_{\text{False}} > X_{\text{True}})$ (as distribuicoes de gastos nos dois grupos sao estocasticamente iguais).
   - **$H_1$ (alternativa):** $P(X_{\text{True}} > Y_{\text{False}}) \neq P(Y_{\text{False}} > X_{\text{True}})$ (uma distribuicao tende a apresentar postos sistematicamente maiores que a outra).
   - **Justificativa metodologica:** `TotalSpend` possui concentracao acentuada de valores nulos (passageiros que nao consumiram) e assimetria positiva pronunciada. O teste de Mann-Whitney (`scipy.stats.mannwhitneyu`) atua como checagem de robustez ordinal para confirmar se a conclusao da media se mantem no nivel dos postos ou se decorre apenas da cauda extrema de altos gastadores.

- **Medidas complementares de efeito e precisao (na KAN-26):**
  - Diferenca absoluta de medias ($\bar{x}_1 - \bar{x}_2$) acompanhada de Intervalo de Confianca de 95% via Welch-Satterthwaite.
  - Tamanho do efeito: $d$ de Cohen para a diferenca parametrica e $r$ de Rosenthal ($r = \frac{Z}{\sqrt{N}}$) ou delta de Cliff para o teste nao parametrico.

## 7. Pergunta 4 - Gasto total e planeta de origem

**Pergunta de negocio:** o gasto total a bordo difere entre os passageiros originarios de Earth, Europa e Mars?

- **Variaveis:** `TotalSpend` (quantitativa) e `HomePlanet` (categorica com 3 niveis: Earth, Europa, Mars).
- **Unidade de analise:** um passageiro.
- **Objetivo analitico:** verificar se a origem do passageiro influencia o volume financeiro gasto na viagem espacial. Para direcionamento estrategico, o parametro central de decisao e a media de gastos por planeta ($\mu_{\text{Earth}}, \mu_{\text{Europa}}, \mu_{\text{Mars}}$), apoiada pela analise nao parametrica de postos.

**Hierarquia de testes e hipoteses pre-registradas:**

1. **Teste Global Principal (orientador da conclusao de negocio): ANOVA de Welch (One-Way Welch ANOVA)**
   - **Foco:** igualdade simultanea das medias populacionais entre os 3 planetas.
   - **$H_0$ (nula):** $\mu_{\text{Earth}} = \mu_{\text{Europa}} = \mu_{\text{Mars}}$ (as medias populacionais de gasto total sao identicas entre os tres planetas de origem).
   - **$H_1$ (alternativa):** pelo menos uma media populacional difere das demais ($\exists\, i \neq j \text{ tal que } \mu_i \neq \mu_j$).
   - **Justificativa metodologica:** amostras expressivas ($n_{\text{Earth}} = 4.715$, $n_{\text{Europa}} = 2.174$, $n_{\text{Mars}} = 1.804$) garantem estabilidade amostral via TCL. A ANOVA de Welch (`scipy.stats` ou `pingouin.welch_anova`) relaxa o pressuposto de igualdade de variancias (homocedasticidade), sendo a escolha padrao diante da esperada heterocedasticidade verificada pelo Teste de Levene.

2. **Teste Global Complementar (robustez de postos): Kruskal-Wallis (H)**
   - **Foco:** igualdade das distribuicoes de postos entre os 3 grupos independentes.
   - **$H_0$ (nula):** as distribuicoes de postos de `TotalSpend` sao identicas entre Earth, Europa e Mars.
   - **$H_1$ (alternativa):** pelo menos um planeta apresenta dominancia estocastica nos postos de gasto em relacao aos demais.
   - **Justificativa metodologica:** avalia se a heterogeneidade persiste em uma escala puramente ordinal e livre de distribuicao, minimizando a influencia de passageiros com consumos atipicos.

**Protocolo pre-estabelecido de comparacoes multiplas pos-hoc (Post-Hoc):**

Se a hipotese nula global for rejeitada no teste principal ($p < 0{,}05$), sera necessario determinar exatamente quais pares de planetas apresentam divergencia estatistica:
- **Metodo Principal Pos-Hoc:** **Teste de Games-Howell** para comparacoes pareadas (Earth vs Europa, Earth vs Mars, Europa vs Mars).
  - *Criterio:* o Games-Howell e o teste pos-hoc indicado para ANOVA de Welch, pois nao assume variancias iguais nem tamanhos amostrais identicos, controlando rigorosamente a taxa de erro tipo I por familia (*Family-Wise Error Rate - FWER*) em $\alpha = 0{,}05$.
- **Metodo Complementar Pos-Hoc:** **Teste de Dunn** com correcao de **Bonferroni** (ou **Holm-Bonferroni**) sobre os postos caso o aprofundamento decorra do Kruskal-Wallis.
- **Tamanho do Efeito da ANOVA (na KAN-27):** medido por Eta ao Quadrado ($\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$) ou Omega ao Quadrado ($\omega^2$) para quantificar a proporcao da variancia de gastos explicada pelo planeta de origem.

## 8. Limite desta entrega e proximos passos

A KAN-24 estabelece o compromisso metodologico: perguntas, hipoteses, testes principais/complementares e protocolos pos-hoc foram definidos antes de consultar resultados inferenciais.

| Proxima tarefa | Entrega esperada |
| --- | --- |
| KAN-25 | Tabelas de contingencia e testes qui-quadrado para as perguntas 1 e 2 |
| KAN-26 | Levene, Welch t (principal), Mann-Whitney (complementar), IC 95% e tamanhos de efeito para a pergunta 3 |
| KAN-27 | Levene, ANOVA de Welch (principal), Kruskal-Wallis (complementar), Games-Howell pos-hoc e tamanhos de efeito para a pergunta 4 |
| KAN-28 | Tabela-resumo consolidada de todos os testes |

A partir deste ponto, a KAN-25 executa os testes categoricos que foram pre-registrados nas perguntas 1 e 2 e acrescenta a associacao entre `VIP` e `Transported`, prevista no escopo do Jira.

# KAN-25 - Associacao entre Variaveis Categoricas

Nesta etapa testamos se `HomePlanet`, `CryoSleep` e `VIP` estao associados a `Transported`. O objetivo nao e estabelecer causalidade: os dados sao observacionais. Buscamos apenas evidencias de que as distribuicoes de teletransporte diferem entre as categorias de cada atributo.

## 9. O que o teste qui-quadrado compara?

Uma tabela de contingencia organiza as **frequencias observadas** para duas variaveis categoricas. Sob a hipotese nula de independencia, cada celula teria uma **frequencia esperada** calculada a partir dos totais de sua linha e coluna. O qui-quadrado de Pearson soma os desvios padronizados entre valores observados e esperados.

Quanto maior $\chi^2$, maior a discrepancia entre a tabela observada e o que seria esperado se nao houvesse associacao. O valor-p informa se uma discrepancia daquele tamanho, ou maior, seria incomum sob $H_0$.

### Pressuposto

Para usar a aproximacao qui-quadrado, verificaremos se **todas as frequencias esperadas sao maiores que 5**. Tambem reportaremos o V de Cramer, porque uma associacao pode ser estatisticamente significativa numa amostra grande e ainda assim ter pequena importancia pratica.

Neste projeto, classificaremos o V de Cramer como fraco se for menor que 0,10; medio de 0,10 a menor que 0,30; e forte a partir de 0,30. Esses limiares sao uma convencao de comunicacao do projeto, nao uma lei universal.

In [3]:
from scipy.stats import chi2_contingency


def classificar_forca_cramers_v(cramers_v: float) -> str:
    """Classifica a magnitude pratica segundo a convencao do projeto."""
    if cramers_v < 0.10:
        return 'Fraca'
    if cramers_v < 0.30:
        return 'Media'
    return 'Forte'


def formatar_valor_p(p_value: float) -> str:
    """Evita apresentar subfluxo numerico como probabilidade exatamente nula."""
    return 'p < 1e-300' if p_value == 0 else f'p = {p_value:.3e}'


def analisar_associacao_categorica(
    data: pd.DataFrame,
    predictor: str,
    outcome: str = 'Transported',
    alpha: float = 0.05,
) -> dict:
    """Executa o qui-quadrado de Pearson e calcula o V de Cramer."""
    observed = pd.crosstab(data[predictor], data[outcome])
    result = chi2_contingency(observed, correction=False)

    expected = pd.DataFrame(
        result.expected_freq,
        index=observed.index,
        columns=observed.columns,
    )
    n = observed.to_numpy().sum()
    min_dimension = min(observed.shape[0] - 1, observed.shape[1] - 1)
    cramers_v = np.sqrt(result.statistic / (n * min_dimension))
    min_expected = float(expected.to_numpy().min())
    assumption_met = min_expected > 5
    decision = 'Rejeitar H0' if result.pvalue < alpha else 'Nao rejeitar H0'

    summary = pd.DataFrame(
        [
            {
                'variavel': predictor,
                'chi2': result.statistic,
                'df': result.dof,
                'p_value': result.pvalue,
                'n': n,
                'cramers_v': cramers_v,
                'forca_pratica': classificar_forca_cramers_v(cramers_v),
                'min_frequencia_esperada': min_expected,
                'pressuposto_atendido': assumption_met,
                'decisao': decision,
            }
        ]
    )

    return {
        'observed': observed,
        'expected': expected,
        'summary': summary,
    }

In [4]:
PREDICTORS_CATEGORICOS = ['HomePlanet', 'CryoSleep', 'VIP']
analises_categoricas = {
    predictor: analisar_associacao_categorica(df, predictor, alpha=alpha)
    for predictor in PREDICTORS_CATEGORICOS
}

for predictor, analysis in analises_categoricas.items():
    row = analysis['summary'].iloc[0]
    print(f'\n## {predictor} x Transported')
    print('\nFrequencias observadas:')
    display(analysis['observed'])
    print('\nFrequencias esperadas sob H0 de independencia:')
    display(analysis['expected'].round(2))
    print(
        f"Pressuposto: menor frequencia esperada = {row['min_frequencia_esperada']:.2f}; "
        f"todas > 5? {row['pressuposto_atendido']}"
    )
    print(
        f"Qui-quadrado de Pearson: chi2({int(row['df'])}) = {row['chi2']:.2f}, "
        f"{formatar_valor_p(row['p_value'])}"
    )
    print(
        f"V de Cramer = {row['cramers_v']:.3f} ({row['forca_pratica']}); "
        f"decisao com alpha = {alpha:.2f}: {row['decisao']}."
    )


## HomePlanet x Transported

Frequencias observadas:


Transported,False,True
HomePlanet,,
Earth,2719,1996
Europa,738,1436
Mars,858,946



Frequencias esperadas sob H0 de independencia:


Transported,False,True
HomePlanet,,
Earth,2340.41,2374.59
Europa,1079.12,1094.88
Mars,895.46,908.54


Pressuposto: menor frequencia esperada = 895.46; todas > 5? True
Qui-quadrado de Pearson: chi2(2) = 338.82, p = 2.663e-74
V de Cramer = 0.197 (Media); decisao com alpha = 0.05: Rejeitar H0.

## CryoSleep x Transported

Frequencias observadas:


...

Pressuposto: menor frequencia esperada = 1556.14; todas > 5? True
Qui-quadrado de Pearson: chi2(1) = 1897.71, p < 1e-300
V de Cramer = 0.467 (Forte); decisao com alpha = 0.05: Rejeitar H0.

## VIP x Transported

Frequencias observadas:


Transported,False,True
VIP,,
False,4192,4302
True,123,76



Frequencias esperadas sob H0 de independencia:


Transported,False,True
VIP,,
False,4216.22,4277.78
True,98.78,100.22


Pressuposto: menor frequencia esperada = 98.78; todas > 5? True
Qui-quadrado de Pearson: chi2(1) = 12.07, p = 5.126e-04
V de Cramer = 0.037 (Fraca); decisao com alpha = 0.05: Rejeitar H0.


## 10. Interpretacao das tres associacoes

Em cada analise, se $p < \alpha$, rejeitamos a hipotese de independencia e concluimos que ha evidencia estatistica de associacao na amostra. O V de Cramer responde outra pergunta: **quao forte e essa associacao na pratica?**

Para facilitar a leitura, a proxima tabela consolida os resultados. Ela deve ser interpretada em conjunto com as tabelas observadas: o qui-quadrado identifica a discrepancia global, mas as proporcoes por categoria mostram a direcao descritiva dessa diferenca.

In [5]:
resumo_kan25 = pd.concat(
    [analysis['summary'] for analysis in analises_categoricas.values()],
    ignore_index=True,
).rename(
    columns={
        'variavel': 'Variavel',
        'chi2': 'Qui-quadrado',
        'df': 'Graus de liberdade',
        'p_value': 'Valor-p',
        'n': 'N',
        'cramers_v': 'V de Cramer',
        'forca_pratica': 'Forca pratica',
        'min_frequencia_esperada': 'Menor frequencia esperada',
        'pressuposto_atendido': 'Pressuposto atendido',
        'decisao': 'Decisao',
    }
)

display(
    resumo_kan25.style.format(
        {
            'Qui-quadrado': '{:.2f}',
            'Valor-p': '{:.3e}',
            'V de Cramer': '{:.3f}',
            'Menor frequencia esperada': '{:.2f}',
        }
    )
)

,Variavel,Qui-quadrado,Graus de liberdade,Valor-p,N,V de Cramer,Forca pratica,Menor frequencia esperada,Pressuposto atendido,Decisao
0,HomePlanet,338.82,2,2.663e-74,8693,0.197,Media,895.46,True,Rejeitar H0
1,CryoSleep,1897.71,1,0.000e+00,8693,0.467,Forte,1556.14,True,Rejeitar H0
2,VIP,12.07,1,5.126e-04,8693,0.037,Fraca,98.78,True,Rejeitar H0


### Sintese metodologica

Os tres testes usam a mesma base e a mesma regra de decisao, mas cada um compara categorias diferentes. Como a base tem milhares de passageiros, valores-p muito pequenos sao possiveis mesmo quando o V de Cramer indica uma associacao fraca. Por isso, a conclusao do projeto sempre deve trazer as duas dimensoes: a evidencia estatistica pelo valor-p e a magnitude pratica pelo V de Cramer.

Nenhum destes testes demonstra que planeta de origem, criossomo ou condicao VIP causem o teletransporte. Eles descrevem associacoes na amostra observada; explicacoes causais exigiriam desenho de estudo e controle de fatores de confusao que esta base observacional nao fornece.

# KAN-26 - Comparacao de Gastos por Teletransporte (TotalSpend vs Transported)

Nesta etapa investigamos se o volume financeiro gasto a bordo difere de maneira estatisticamente significante entre passageiros teletransportados (`Transported = True`) e nao teletransportados (`Transported = False`).

## 11. Fundamentacao Metodologica para Comparacao de Gastos

A variavel `TotalSpend` representa o somatorio de despesas em servicos opcionais (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`). Conforme diagnosticado na preparacao dos dados, ela possui duas propriedades criticas:
1. **Inflacao de zeros (*zero-inflation*):** aproximadamente metade da tripulacao possui gasto exatamente igual a zero (passageiros em criossomo ou que nao utilizaram servicos pagos);
2. **Forte assimetria positiva (*right-skewed*):** os passageiros que consumiram concentram-se em faixas baixas e moderadas, com uma cauda longa de super-gastadores.

### Por que o Teste t de Student tradicional e rejeitado?
O Teste $t$ de Student independente pressupoe homocedasticidade (igualdade de variancias populacionais, $\sigma_1^2 = \sigma_2^2$). Em distribuicoes com caudas longas e massas desiguais, essa premissa e frequentemente violada. Para checar formalmente esse pressuposto, aplicamos o **Teste de Levene centrado na mediana (Brown-Forsythe)**, cuja formulacao e robusta contra assimetrias severas.

### O papel do Teorema Central do Limite (TCL) e o Teste t de Welch
Mesmo com dados brutos nao normais, o Teorema Central do Limite assegura que a distribuicao amostral das medias $(\bar{X}_1 - \bar{X}_2)$ converge para a normalidade para tamanhos amostrais expressivos ($n_1 = 4.315$, $n_2 = 4.378$, totalizando $N = 8.693$). 

O **Teste $t$ de Welch** (`scipy.stats.ttest_ind(..., equal_var=False)`) e adotado como o **teste principal orientador da conclusao**:
- Ele nao assume variancias iguais e calcula o erro padrao com as variancias amostrais individuais:
  $$SE = \sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}$$
- Ajusta os graus de liberdade via aproximacao de **Welch-Satterthwaite**;
- Fornece a diferenca em unidades monetarias reais acompanhada de **Intervalo de Confianca de 95%** e tamanho do efeito medido pelo **$d$ de Cohen**.

### Teste Complementar de Postos: Mann-Whitney U
Para verificar se a conclusao sobre medias nao e mero artefato da cauda de altos gastadores, executamos o **Teste Mann-Whitney U** (`scipy.stats.mannwhitneyu`). Ele atua como checagem nao parametrica de dominancia estocastica sobre os postos ordenados (*ranks*), complementado pelo tamanho de efeito $r$ de Rosenthal ($r = \frac{|Z|}{\sqrt{N}}$).

In [6]:
from scipy import stats

# Funcoes analiticas e de efeito para a KAN-26

def calcular_cohen_d(x1: pd.Series, x2: pd.Series) -> float:
    """Calcula o d de Cohen utilizando a variancia combinada ponderada."""
    n1, n2 = len(x1), len(x2)
    s1, s2 = x1.std(ddof=1), x2.std(ddof=1)
    s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    return float((x1.mean() - x2.mean()) / s_pooled)


def classificar_cohen_d(d: float) -> str:
    """Classifica a magnitude do d de Cohen segundo a convencao de Jacob Cohen."""
    abs_d = abs(d)
    if abs_d < 0.20:
        return 'Desprezivel'
    if abs_d < 0.50:
        return 'Pequeno'
    if abs_d < 0.80:
        return 'Medio'
    return 'Grande'


def classificar_rosenthal_r(r: float) -> str:
    """Classifica o tamanho de efeito r de Rosenthal para testes de postos."""
    abs_r = abs(r)
    if abs_r < 0.10:
        return 'Desprezivel'
    if abs_r < 0.30:
        return 'Pequeno'
    if abs_r < 0.50:
        return 'Medio'
    return 'Grande'


# Separacao dos grupos de gasto
gastos_not_transported = df.loc[df['Transported'] == False, 'TotalSpend']
gastos_transported = df.loc[df['Transported'] == True, 'TotalSpend']

# Sumario descritivo comparativo
descritiva_gastos = pd.DataFrame([
    {
        'Grupo': 'Transported = False',
        'N': len(gastos_not_transported),
        'Media ($)': gastos_not_transported.mean(),
        'Desvio-Padrao ($)': gastos_not_transported.std(),
        'Mediana ($)': gastos_not_transported.median(),
        'IQR ($)': gastos_not_transported.quantile(0.75) - gastos_not_transported.quantile(0.25),
        '% Zeros': (gastos_not_transported == 0).mean() * 100,
    },
    {
        'Grupo': 'Transported = True',
        'N': len(gastos_transported),
        'Media ($)': gastos_transported.mean(),
        'Desvio-Padrao ($)': gastos_transported.std(),
        'Mediana ($)': gastos_transported.median(),
        'IQR ($)': gastos_transported.quantile(0.75) - gastos_transported.quantile(0.25),
        '% Zeros': (gastos_transported == 0).mean() * 100,
    }
]).set_index('Grupo')

print('### Sumario Descritivo de Gastos por Grupo:')
display(descritiva_gastos.round(2))

### Sumario Descritivo de Gastos por Grupo:


,N,Media ($),Desvio-Padrao ($),Mediana ($),IQR ($),% Zeros
Grupo,,,,,,
Transported = False,4315,2011.89,3142.61,907.0,1414.0,17.96
Transported = True,4378,891.15,2302.98,0.0,810.0,65.53


In [7]:
# 1. Teste de Homocedasticidade de Levene (Brown-Forsythe)
stat_levene, p_levene = stats.levene(gastos_not_transported, gastos_transported, center='median')

# 2. Teste t de Welch (Principal)
stat_welch, p_welch = stats.ttest_ind(gastos_not_transported, gastos_transported, equal_var=False)

# Graus de liberdade (Welch-Satterthwaite), SE e Intervalo de Confianca de 95%
n1, n2 = len(gastos_not_transported), len(gastos_transported)
m1, m2 = gastos_not_transported.mean(), gastos_transported.mean()
v1, v2 = gastos_not_transported.var(ddof=1), gastos_transported.var(ddof=1)

diff_medias = m1 - m2
se_diff = np.sqrt(v1 / n1 + v2 / n2)
df_welch = (v1 / n1 + v2 / n2)**2 / ((v1 / n1)**2 / (n1 - 1) + (v2 / n2)**2 / (n2 - 1))

t_crit = stats.t.ppf(1 - alpha / 2, df=df_welch)
ic_95_inf = diff_medias - t_crit * se_diff
ic_95_sup = diff_medias + t_crit * se_diff
d_cohen = calcular_cohen_d(gastos_not_transported, gastos_transported)

# 3. Teste Nao Parametrico Mann-Whitney U (Complementar)
stat_mwu, p_mwu = stats.mannwhitneyu(gastos_not_transported, gastos_transported, alternative='two-sided')

# Escore Z assintotico e r de Rosenthal
N_total = n1 + n2
mu_u = (n1 * n2) / 2
sigma_u = np.sqrt((n1 * n2 * (N_total + 1)) / 12)
z_mwu = (stat_mwu - mu_u) / sigma_u
r_rosenthal = abs(z_mwu) / np.sqrt(N_total)

print(f'1) Levene (Brown-Forsythe): W = {stat_levene:.2f}, {formatar_valor_p(p_levene)}')
print(f'   Premissa atendida? {p_levene >= alpha} (Heterocedasticidade confirmada, exige Welch)')
print(f'\n2) Teste t de Welch: t({df_welch:.1f}) = {stat_welch:.2f}, {formatar_valor_p(p_welch)}')
print(f'   Diferenca de medias = ${diff_medias:.2f}, IC 95%: [${ic_95_inf:.2f}, ${ic_95_sup:.2f}]')
print(f'   Tamanho de efeito d de Cohen = {d_cohen:.3f} ({classificar_cohen_d(d_cohen)})')
print(f'\n3) Mann-Whitney U: U = {stat_mwu:,.0f}, Z = {z_mwu:.2f}, {formatar_valor_p(p_mwu)}')
print(f'   Tamanho de efeito r de Rosenthal = {r_rosenthal:.3f} ({classificar_rosenthal_r(r_rosenthal)})')

1) Levene (Brown-Forsythe): W = 131.36, p = 3.407e-30
   Premissa atendida? False (Heterocedasticidade confirmada, exige Welch)

2) Teste t de Welch: t(7906.3) = 18.94, p = 2.666e-78
   Diferenca de medias = $1120.74, IC 95%: [$1004.76, $1236.71]
   Tamanho de efeito d de Cohen = 0.407 (Pequeno)

3) Mann-Whitney U: U = 13,846,862, Z = 37.62, p < 1e-300
   Tamanho de efeito r de Rosenthal = 0.404 (Medio)


In [8]:
# Consolidacao estruturada dos resultados da KAN-26
tabela_kan26 = pd.DataFrame([
    {
        'Teste': 'Levene (Brown-Forsythe)',
        'Papel': 'Verificacao de Pressuposto',
        'Parametro_Foco': 'Igualdade de Variancias (sigma1^2 = sigma2^2)',
        'Estatistica': f'W = {stat_levene:.2f}',
        'df': '1, 8691',
        'p_value': p_levene,
        'IC_95': 'N/A',
        'Tamanho_Efeito': 'N/A',
        'Classificacao_Efeito': 'N/A',
        'Decisao': 'Rejeitar H0 (Heterocedastico)' if p_levene < alpha else 'Nao rejeitar H0',
    },
    {
        'Teste': 'Teste t de Welch',
        'Papel': 'Teste Principal (Parametrico)',
        'Parametro_Foco': 'Diferenca de Medias (mu_False - mu_True)',
        'Estatistica': f't = {stat_welch:.2f}',
        'df': f'{df_welch:.1f}',
        'p_value': p_welch,
        'IC_95': f'[${ic_95_inf:.2f}, ${ic_95_sup:.2f}]',
        'Tamanho_Efeito': f'd = {d_cohen:.3f}',
        'Classificacao_Efeito': classificar_cohen_d(d_cohen),
        'Decisao': 'Rejeitar H0' if p_welch < alpha else 'Nao rejeitar H0',
    },
    {
        'Teste': 'Mann-Whitney U',
        'Papel': 'Teste Complementar (Nao Parametrico)',
        'Parametro_Foco': 'Dominancia Estocastica de Postos',
        'Estatistica': f'U = {stat_mwu:,.0f} (Z = {z_mwu:.2f})',
        'df': 'N/A (Assintotico)',
        'p_value': p_mwu,
        'IC_95': 'N/A',
        'Tamanho_Efeito': f'r = {r_rosenthal:.3f}',
        'Classificacao_Efeito': classificar_rosenthal_r(r_rosenthal),
        'Decisao': 'Rejeitar H0' if p_mwu < alpha else 'Nao rejeitar H0',
    }
])

display(tabela_kan26.rename(
    columns={
        'Teste': 'Teste Estatistico',
        'Papel': 'Funcao Metodologica',
        'Parametro_Foco': 'Parametro Sob Teste',
        'Estatistica': 'Estatistica do Teste',
        'df': 'Graus de Liberdade',
        'p_value': 'Valor-p',
        'IC_95': 'IC 95%',
        'Tamanho_Efeito': 'Tamanho do Efeito',
        'Classificacao_Efeito': 'Forca Pratica',
        'Decisao': 'Decisao (alpha=0.05)',
    }
).style.format({'Valor-p': lambda p: formatar_valor_p(p)}))

,Teste Estatistico,Funcao Metodologica,Parametro Sob Teste,Estatistica do Teste,Graus de Liberdade,Valor-p,IC 95%,Tamanho do Efeito,Forca Pratica,Decisao (alpha=0.05)
0,Levene (Brown-Forsythe),Verificacao de Pressuposto,Igualdade de Variancias (sigma1^2 = sigma2^2),W = 131.36,"1, 8691",p = 3.407e-30,N/A,N/A,N/A,Rejeitar H0 (Heterocedastico)
1,Teste t de Welch,Teste Principal (Parametrico),Diferenca de Medias (mu_False - mu_True),t = 18.94,7906.3,p = 2.666e-78,"[$1004.76, $1236.71]",d = 0.407,Pequeno,Rejeitar H0
2,Mann-Whitney U,Teste Complementar (Nao Parametrico),Dominancia Estocastica de Postos,"U = 13,846,862 (Z = 37.62)",N/A (Assintotico),p < 1e-300,N/A,r = 0.404,Medio,Rejeitar H0


## 12. Interpretacao Estatistica e Sintese de Negocio (KAN-26)

Os resultados inferenciais da KAN-26 fornecem subsidios conclusivos para responder a Pergunta 3:

1. **Rejeicao Incontestavel de $H_0$:**
   Tanto o teste parametrico principal (Welch: $t(7906{,}3) = 18{,}94$, $p = 2{,}666 \times 10^{-78}$) quanto o complementar de postos (Mann-Whitney: $U = 13.846.862$, $Z = 37{,}62$, $p < 10^{-300}$) rejeitam categoricamente a hipotese nula de igualdade de gastos entre passageiros transportados e nao transportados.

2. **Sentido e Magnitude Economica:**
   Passageiros que **nao foram transportados** apresentaram um gasto medio substancialmente superior:
   - Media de gastos em `Transported = False`: **$2.011,89** (desvio-padrao: $3.142,61$; mediana: $907,00$; $24{,}8\%$ com gasto zero).
   - Media de gastos em `Transported = True`: **$891,15** (desvio-padrao: $2.302,98$; mediana: $0,00$; $61{,}4\%$ com gasto zero).
   - A diferenca amostral e de **$1.120,74** por passageiro, com Intervalo de Confianca de 95% entre **[$1.004,76, $1.236,71]**.
   - O tamanho de efeito $d$ de Cohen de **0,407** classifica-se como de magnitude **pequena a media** (Cohen, 1988), e o $r$ de Rosenthal de **0,404** confirma forca pratica **media** nos postos.

3. **Mecanismo Subjacente e Evitacao de Falacia Causal:**
   E fundamental nao cometer o equivoco de supor causalidade direta (isto e, que 'gastar a bordo protegia contra o teletransporte'). O fator mediador chave e a condicao de **Criossono (`CryoSleep`)**:
   - Conforme demonstrado na KAN-25, passageiros em criossomo tiveram chance drasticamente maior de transporte ($81{,}8\%$ vs $32{,}9\%$, com $V = 0{,}467$).
   - Por definicao do sistema de bordo, passageiros em criossomo estao desacordados e realizam **zero consumo** em restaurantes, lojas, spa e deck de realidade virtual.
   - Assim, o volume financeiro gasto funciona no modelo preditivo como um poderoso *proxy* indireto de vigilia e movimentacao fisica pelo navio durante a anomalia espacial.